In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install timm albumentations onnx onnxscript -q
import os
import cv2
import random
import numpy as np
import pandas as pd

from glob import glob
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

import matplotlib.pyplot as plt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 15.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 64.0 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires n

In [2]:
PALM_PATH = "/kaggle/input/datasets/olankadhim/multimodal-biometric-dataset-mulb/MULB dataset/hand dataset"

IRIS_PATH = "/kaggle/input/datasets/olankadhim/multimodal-biometric-dataset-mulb/MULB dataset/iris dataset"

IMG_SIZE = 224

In [3]:
def clahe_enhancement(img):

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    l,a,b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    cl = clahe.apply(l)

    merged = cv2.merge((cl,a,b))

    final = cv2.cvtColor(
        merged,
        cv2.COLOR_LAB2BGR
    )

    return final

In [4]:
train_transform = A.Compose([
    A.Resize(224,224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Rotate(limit=15,p=0.5),
    A.GaussianBlur(p=0.3),
    A.Normalize(),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(224,224),
    A.Normalize(),
    ToTensorV2()
])

In [5]:
class MULBTripletDataset(Dataset):

    def __init__(self, hand_root, iris_root, transform=None):

        self.transform = transform

        self.subjects = []

        self.hand_images = {}
        self.iris_images = {}

        hand_subjects = os.listdir(hand_root)

        for subject in hand_subjects:

            hand_files = (
                glob(os.path.join(hand_root, subject, "*.jpg")) +
                glob(os.path.join(hand_root, subject, "*.png")) +
                glob(os.path.join(hand_root, subject, "*.bmp")) +
                glob(os.path.join(hand_root, subject, "*.jpeg"))
            )

            iris_files = (
                glob(os.path.join(iris_root, subject, "*.jpg")) +
                glob(os.path.join(iris_root, subject, "*.png")) +
                glob(os.path.join(iris_root, subject, "*.bmp")) +
                glob(os.path.join(iris_root, subject, "*.jpeg"))
            )

            # keep valid subjects only
            if len(hand_files) > 0 and len(iris_files) > 0:

                self.subjects.append(subject)

                self.hand_images[subject] = hand_files
                self.iris_images[subject] = iris_files

        print(f"Total Valid Subjects: {len(self.subjects)}")

    def __len__(self):
        return len(self.subjects) * 20

    def read_image(self, path):

        img = cv2.imread(path)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = clahe_enhancement(img)

        if self.transform:
            img = self.transform(image=img)["image"]

        return img

    def __getitem__(self, idx):

        anchor_subject = random.choice(self.subjects)

        positive_subject = anchor_subject

        negative_subject = random.choice(self.subjects)

        while negative_subject == anchor_subject:
            negative_subject = random.choice(self.subjects)

        anchor_hand = random.choice(self.hand_images[anchor_subject])
        positive_hand = random.choice(self.hand_images[positive_subject])
        negative_hand = random.choice(self.hand_images[negative_subject])

        anchor_iris = random.choice(self.iris_images[anchor_subject])
        positive_iris = random.choice(self.iris_images[positive_subject])
        negative_iris = random.choice(self.iris_images[negative_subject])

        sample = {

            "anchor_palm": self.read_image(anchor_hand),
            "positive_palm": self.read_image(positive_hand),
            "negative_palm": self.read_image(negative_hand),

            "anchor_iris": self.read_image(anchor_iris),
            "positive_iris": self.read_image(positive_iris),
            "negative_iris": self.read_image(negative_iris)
        }

        return sample

In [6]:
train_dataset = MULBTripletDataset(
    PALM_PATH,
    IRIS_PATH,
    transform=train_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

Total Valid Subjects: 169


In [7]:
class QualityAssessment:

    def blur_score(self, img):
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        return cv2.Laplacian(gray, cv2.CV_64F).var()

    def brightness_score(self, img):
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        return hsv[:,:,2].mean()

    def contrast_score(self, img):
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        return gray.std()

    def overall_quality(self, img):

        blur = self.blur_score(img)
        bright = self.brightness_score(img)
        contrast = self.contrast_score(img)

        quality = (
            0.4 * blur +
            0.3 * bright +
            0.3 * contrast
        )

        return quality

In [8]:
class QualityCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            'resnet18',
            pretrained=True,
            num_classes=0
        )

        self.fc = nn.Linear(512,1)

    def forward(self,x):

        feat = self.backbone(x)
        q = torch.sigmoid(self.fc(feat))

        return q

In [9]:
class SpoofDetector(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = timm.create_model(
            'resnet18',
            pretrained=True,
            num_classes=0
        )

        self.classifier = nn.Sequential(
            nn.Linear(512,128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128,1)
        )

    def forward(self,x):

        feat = self.encoder(x)
        spoof = torch.sigmoid(self.classifier(feat))

        return spoof

In [10]:
class EmbeddingNetwork(nn.Module):

    def __init__(self, backbone='resnet18'):
        super().__init__()

        self.backbone = timm.create_model(
            backbone,
            pretrained=True,
            num_classes=0
        )

        self.embedding = nn.Sequential(
            nn.Linear(512,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256,128)
        )

    def forward(self,x):

        feat = self.backbone(x)

        emb = self.embedding(feat)

        emb = F.normalize(emb,p=2,dim=1)

        return emb

In [11]:
triplet_loss = nn.TripletMarginLoss(
    margin=1.0,
    p=2
)

In [12]:
class AdaptiveFusion(nn.Module):

    def __init__(self, embedding_dim=128):

        super().__init__()

        self.hand_gate = nn.Sequential(

            nn.Linear(embedding_dim, embedding_dim),

            nn.Sigmoid()
        )

        self.iris_gate = nn.Sequential(

            nn.Linear(embedding_dim, embedding_dim),

            nn.Sigmoid()
        )

        self.fusion_layer = nn.Sequential(

            nn.Linear(embedding_dim, embedding_dim),

            nn.ReLU(),

            nn.Dropout(0.2)
        )

    def forward(
        self,
        hand_emb,
        iris_emb,
        hand_quality,
        iris_quality,
        hand_spoof,
        iris_spoof
    ):

        gh = self.hand_gate(hand_emb)

        gi = self.iris_gate(iris_emb)

        hand_weight = hand_quality * (1 - hand_spoof)

        iris_weight = iris_quality * (1 - iris_spoof)

        fused = (

            hand_weight * hand_emb * gh +

            iris_weight * iris_emb * gi
        )

        fused = self.fusion_layer(fused)

        fused = F.normalize(fused,p=2,dim=1)

        return fused

In [13]:
class TrustScoring:

    def calculate(
        self,
        similarity,
        quality,
        spoof_prob
    ):

        trust = (
            0.5 * similarity +
            0.3 * quality +
            0.2 * (1 - spoof_prob)
        )

        return trust

In [14]:
class MultimodalVerificationSystem(nn.Module):

    def __init__(self):
        super().__init__()

        self.palm_encoder = EmbeddingNetwork()
        self.iris_encoder = EmbeddingNetwork()

        self.quality_net = QualityCNN()

        self.spoof_detector = SpoofDetector()

        self.fusion = AdaptiveFusion()

    def forward(self,palm_img,iris_img):

        palm_emb = self.palm_encoder(palm_img)
        iris_emb = self.iris_encoder(iris_img)

        palm_quality = self.quality_net(palm_img)
        iris_quality = self.quality_net(iris_img)

        palm_spoof = self.spoof_detector(palm_img)
        iris_spoof = self.spoof_detector(iris_img)

        fused = self.fusion(
            palm_emb,
            iris_emb,
            palm_quality,
            iris_quality,
            palm_spoof,
            iris_spoof
        )

        return fused

In [15]:
DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

model = MultimodalVerificationSystem().to(DEVICE)

triplet_loss = nn.TripletMarginLoss(
    margin=1.0,
    p=2
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4
)

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

In [ ]:
EPOCHS = 40

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    loop = tqdm(train_loader)

    for batch in loop:

        anchor_palm = batch['anchor_palm'].to(DEVICE)
        positive_palm = batch['positive_palm'].to(DEVICE)
        negative_palm = batch['negative_palm'].to(DEVICE)

        anchor_iris = batch['anchor_iris'].to(DEVICE)
        positive_iris = batch['positive_iris'].to(DEVICE)
        negative_iris = batch['negative_iris'].to(DEVICE)

        za = model(anchor_palm, anchor_iris)
        zp = model(positive_palm, positive_iris)
        zn = model(negative_palm, negative_iris)

        loss = triplet_loss(za,zp,zn)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        loop.set_description(f'Epoch {epoch}')
        loop.set_postfix(loss=loss.item())

    print(f'Average Loss: {total_loss/len(train_loader)}')

Epoch 0:   3%|▎         | 23/845 [01:38<57:45,  4.22s/it, loss=1.04]   

In [ ]:
def compute_distance(a,b):
    return torch.norm(a-b,p=2,dim=1)

def load_image(path):

    img = cv2.imread(path)

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    img = clahe_enhancement(img)

    img = val_transform(
        image=img
    )["image"]

    return img.unsqueeze(0)

In [ ]:
# =====================================
# COMPLETE INFERENCE PIPELINE
# =====================================

model.eval()

# -------------------------------------
# IMAGE LOADING FUNCTION
# -------------------------------------

def load_image(path):

    img = cv2.imread(path)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = clahe_enhancement(img)

    img = val_transform(image=img)["image"]

    return img.unsqueeze(0)


# -------------------------------------
# SELECT TEST SUBJECT
# -------------------------------------

# SAME PERSON TEST

subject = train_dataset.subjects[0]

palm1_path = train_dataset.hand_images[subject][0]
iris1_path = train_dataset.iris_images[subject][0]

palm2_path = train_dataset.hand_images[subject][1]
iris2_path = train_dataset.iris_images[subject][1]


# -------------------------------------
# LOAD IMAGES
# -------------------------------------

palm1 = load_image(palm1_path).to(DEVICE)
iris1 = load_image(iris1_path).to(DEVICE)

palm2 = load_image(palm2_path).to(DEVICE)
iris2 = load_image(iris2_path).to(DEVICE)


# -------------------------------------
# DISTANCE FUNCTION
# -------------------------------------

def compute_distance(a, b):

    return torch.norm(a - b, p=2, dim=1)


# -------------------------------------
# VERIFICATION FUNCTION
# -------------------------------------

THRESHOLD = 0.5

def verify(distance):

    if distance.item() < THRESHOLD:

        return "Same Person Verified"

    else:

        return "Different Person"


# -------------------------------------
# INFERENCE
# -------------------------------------

with torch.no_grad():

    emb1 = model(palm1, iris1)

    emb2 = model(palm2, iris2)

    distance = compute_distance(emb1, emb2)

    result = verify(distance)

    print("Distance:", distance.item())

    print("Result:", result)

In [ ]:
NUM_GENUINE = 300
NUM_IMPOSTOR = 300

all_labels = []
all_scores = []

all_trust_scores = []
all_spoof_scores = []

model.eval()

with torch.no_grad():

    # Genuine pairs

    for _ in range(NUM_GENUINE):

        subject = random.choice(
            train_dataset.subjects
        )

        if len(train_dataset.hand_images[subject]) < 2:
            continue

        palm1 = load_image(
            random.choice(
                train_dataset.hand_images[subject]
            )
        ).to(DEVICE)

        iris1 = load_image(
            random.choice(
                train_dataset.iris_images[subject]
            )
        ).to(DEVICE)

        palm2 = load_image(
            random.choice(
                train_dataset.hand_images[subject]
            )
        ).to(DEVICE)

        iris2 = load_image(
            random.choice(
                train_dataset.iris_images[subject]
            )
        ).to(DEVICE)

        emb1 = model(palm1,iris1)
        emb2 = model(palm2,iris2)

        distance = compute_distance(
            emb1,
            emb2
        ).item()

        similarity = 1/(1+distance)

        quality = (
            model.quality_net(palm1).item() +
            model.quality_net(iris1).item()
        ) / 2

        spoof = (
            model.spoof_detector(palm1).item() +
            model.spoof_detector(iris1).item()
        ) / 2

        trust = (
            0.5*similarity +
            0.3*quality +
            0.2*(1-spoof)
        )

        all_labels.append(1)
        all_scores.append(similarity)

        all_trust_scores.append(trust)
        all_spoof_scores.append(spoof)

    # Impostor pairs

    for _ in range(NUM_IMPOSTOR):

        s1 = random.choice(
            train_dataset.subjects
        )

        s2 = random.choice(
            train_dataset.subjects
        )

        while s1 == s2:
            s2 = random.choice(
                train_dataset.subjects
            )

        palm1 = load_image(
            random.choice(
                train_dataset.hand_images[s1]
            )
        ).to(DEVICE)

        iris1 = load_image(
            random.choice(
                train_dataset.iris_images[s1]
            )
        ).to(DEVICE)

        palm2 = load_image(
            random.choice(
                train_dataset.hand_images[s2]
            )
        ).to(DEVICE)

        iris2 = load_image(
            random.choice(
                train_dataset.iris_images[s2]
            )
        ).to(DEVICE)

        emb1 = model(palm1,iris1)
        emb2 = model(palm2,iris2)

        distance = compute_distance(
            emb1,
            emb2
        ).item()

        similarity = 1/(1+distance)

        quality = (
            model.quality_net(palm1).item() +
            model.quality_net(iris1).item()
        ) / 2

        spoof = (
            model.spoof_detector(palm1).item() +
            model.spoof_detector(iris1).item()
        ) / 2

        trust = (
            0.5*similarity +
            0.3*quality +
            0.2*(1-spoof)
        )

        all_labels.append(0)
        all_scores.append(similarity)

        all_trust_scores.append(trust)
        all_spoof_scores.append(spoof)

In [ ]:
fpr,tpr,thresholds = roc_curve(
    all_labels,
    all_scores
)

fnr = 1 - tpr

eer_idx = np.nanargmin(
    np.abs(fnr-fpr)
)

eer = fpr[eer_idx]

threshold = thresholds[eer_idx]

preds = [
    1 if s>=threshold else 0
    for s in all_scores
]

acc = accuracy_score(
    all_labels,
    preds
)

prec = precision_score(
    all_labels,
    preds
)

rec = recall_score(
    all_labels,
    preds
)

f1 = f1_score(
    all_labels,
    preds
)

auc = roc_auc_score(
    all_labels,
    all_scores
)

cm = confusion_matrix(
    all_labels,
    preds
)

TN,FP,FN,TP = cm.ravel()

FAR = FP/(FP+TN+1e-8)

FRR = FN/(FN+TP+1e-8)

print("Accuracy :",acc)
print("Precision:",prec)
print("Recall   :",rec)
print("F1 Score :",f1)
print("ROC-AUC  :",auc)

print("FAR      :",FAR)
print("FRR      :",FRR)
print("EER      :",eer)

print("Trust Score:",
      np.mean(all_trust_scores))

print("Spoof Score:",
      np.mean(all_spoof_scores))

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(cm)

plt.colorbar()

plt.title("Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

for i in range(2):
    for j in range(2):

        plt.text(
            j,
            i,
            str(cm[i,j]),
            ha='center'
        )

plt.show()

In [ ]:
plt.figure(figsize=(7,6))

plt.plot(fpr,tpr)

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.grid()

plt.show()

In [ ]:
# =====================================
# INSTALL REQUIRED PACKAGES
# =====================================

!pip install onnx
!pip install onnxscript


# =====================================
# IMPORT LIBRARIES
# =====================================

import torch
import onnx


# =====================================
# SET MODEL TO EVAL MODE
# =====================================

model.eval()


# =====================================
# CREATE DUMMY INPUTS
# =====================================

sample_palm = torch.randn(
    1,
    3,
    224,
    224
).to(DEVICE)

sample_iris = torch.randn(
    1,
    3,
    224,
    224
).to(DEVICE)


# =====================================
# EXPORT TO ONNX
# =====================================

torch.onnx.export(

    model,

    (sample_palm, sample_iris),

    "multimodal_biometric.onnx",

    export_params=True,

    opset_version=11,

    do_constant_folding=True,

    input_names=['palm_input', 'iris_input'],

    output_names=['embedding_output'],

    dynamic_axes={

        'palm_input': {0: 'batch_size'},

        'iris_input': {0: 'batch_size'},

        'embedding_output': {0: 'batch_size'}
    }
)

print("ONNX Model Exported Successfully")

In [ ]:
onnx_model = onnx.load(
    "multimodal_biometric.onnx"
)

onnx.checker.check_model(
    onnx_model
)

print("ONNX Model Verified")